In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Mandir Marg, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Benzene,Toluene,RH,WS,WD,BP,Xylene,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,171.52,222.91,14.36,70.09,48.96,57.55,3.26,1.10,...,NaN,NaN,86.51,0.66,258.62,986.00,NaN,12.38,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,175.56,223.79,16.14,73.54,52.23,73.86,3.59,1.69,...,NaN,NaN,87.49,0.77,250.28,986.00,NaN,12.52,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,261.76,331.95,34.82,120.61,92.46,84.28,6.76,2.58,...,NaN,NaN,87.88,0.35,239.62,986.00,NaN,13.95,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,277.09,331.33,38.18,149.39,110.51,86.37,7.33,2.40,...,NaN,NaN,89.29,0.90,157.10,986.00,NaN,14.66,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,174.28,225.92,17.63,90.14,61.92,70.89,5.29,1.28,...,NaN,NaN,87.90,1.17,129.47,986.00,NaN,13.68,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,313.33,416.21,50.18,46.48,64.12,78.84,53.74,1.23,...,19.01,75.71,70.78,0.52,255.89,986.07,NaN,17.77,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,275.71,384.33,123.73,51.96,138.17,87.46,51.55,1.14,...,18.37,122.90,75.24,0.46,238.83,985.92,NaN,17.55,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,212.46,317.54,32.67,38.39,46.41,75.00,59.13,1.60,...,9.30,44.59,73.77,0.44,236.16,986.34,NaN,17.52,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,218.97,312.08,56.01,25.50,59.11,79.63,69.07,2.21,...,0.01,10.19,73.43,0.44,242.95,986.55,NaN,17.45,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date  PM2.5    PM10     NO     NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   51.8  222.91  14.36   70.09   48.96   
1  02-01-2025 00:00  03-01-2025 00:00   51.8  223.79  16.14   73.54   52.23   
2  03-01-2025 00:00  04-01-2025 00:00   51.8  114.04  34.82  120.61   92.46   
3  04-01-2025 00:00  05-01-2025 00:00   51.8  114.04  38.18  149.39  110.51   
4  05-01-2025 00:00  06-01-2025 00:00   51.8  225.92  17.63   90.14   61.92   

      NH3   SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD     BP  \
0  57.550  3.26  1.10  38.70     4.49    25.57  86.51  0.66  258.62  986.0   
1  73.860  3.59  1.69  37.45     4.49    25.57  87.49  0.77  250.28  986.0   
2  29.695  6.76  0.80  47.35     4.49    25.57  87.88  0.35  239.62  986.0   
3  29.695  7.33  0.80  43.16     4.49    25.57  89.29  0.90  157.10  986.0   
4  70.890  5.29  1.28  23.32     4.49    25.57  87.90  1.17  129.47  986.0   

       AT  TOT-RF  
0  24.935    

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df.to_excel('mandirmarg2025.xlsx', index=False)